In [30]:
import re
import numpy as np
import tensorflow as tf
import os

os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.text import Tokenizer

import kagglehub

## 1. Data

In [31]:
path = kagglehub.dataset_download("rohitgr/wikitext")
wikitext2_path = os.path.join(path, "wikitext-2")

def load_file(base_path, filename):
    with open(os.path.join(base_path, filename), "r", encoding="utf-8") as f:
        return f.read()

train_text = load_file(wikitext2_path, "wiki.train.tokens")
val_text = load_file(wikitext2_path, "wiki.valid.tokens")

## 2. Data Cleaning

In [32]:
def clean_text(text):
    text = text.lower()

    # poista wiki-otsikot
    text = re.sub(r"=+.*?=+", "", text)

    # säilytä perusvälimerkit (tärkeää kielimallille)
    text = re.sub(r"[^a-zA-Z0-9\s\.,']", "", text)

    text = re.sub(r"\s+", " ", text)
    return text

train_text = clean_text(train_text)
val_text = clean_text(val_text)

train_text = train_text[:500_000]
val_text = val_text[:100_000]

## 2. Tokenizer

In [33]:
vocab_size_limit = 15000

tokenizer = Tokenizer(
    num_words=vocab_size_limit,
    oov_token="<OOV>",
    filters=""
)

tokenizer.fit_on_texts([train_text])

sequence = tokenizer.texts_to_sequences([train_text])[0]

word_index = tokenizer.word_index
vocab_size = min(vocab_size_limit, len(word_index) + 1)

index_to_word = {i: w for w, i in word_index.items()}
index_to_word[0] = "<PAD>"
index_to_word[word_index["<OOV>"]] = "<OOV>"

seq_length = 30

## 3. Dataset

In [34]:
X, y = [], []

for i in range(0, len(sequence) - seq_length, 2):  # stride vähentää overfittiä
    X.append(sequence[i:i+seq_length])
    y.append(sequence[i+seq_length])

X = np.array(X)
y = np.array(y)

# shuffle
idx = np.random.permutation(len(X))
X = X[idx]
y = y[idx]

split = int(0.9 * len(X))

X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

## 4. Callback

In [35]:
callback = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

## 5. LSTM Baseline model

In [36]:
lstm_model = models.Sequential([
    layers.Embedding(vocab_size, 256),
    layers.Bidirectional(layers.LSTM(256, return_sequences=True)),
    layers.Dropout(0.3),
    layers.Bidirectional(layers.LSTM(256)),
    layers.Dense(vocab_size, activation="softmax")
])

lstm_model.compile(
    loss="sparse_categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

lstm_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=64,
    callbacks=[callback]
)

Epoch 1/8
658/658 [==============================] - 16s 20ms/step - loss: 6.8069 - accuracy: 0.0886 - val_loss: 6.5899 - val_accuracy: 0.1048
Epoch 2/8
658/658 [==============================] - 7s 11ms/step - loss: 6.1688 - accuracy: 0.1234 - val_loss: 6.5737 - val_accuracy: 0.1162
Epoch 3/8
658/658 [==============================] - 7s 10ms/step - loss: 5.8931 - accuracy: 0.1347 - val_loss: 6.6179 - val_accuracy: 0.1190
Epoch 4/8
658/658 [==============================] - 6s 10ms/step - loss: 5.6693 - accuracy: 0.1507 - val_loss: 6.6927 - val_accuracy: 0.1228
Epoch 5/8
658/658 [==============================] - 6s 10ms/step - loss: 5.4626 - accuracy: 0.1634 - val_loss: 6.8066 - val_accuracy: 0.1228


## 6. Sampling utility

In [37]:
def sample_probs(preds, temperature=1.0):
    preds = np.asarray(preds).astype("float64")

    preds = np.log(preds + 1e-8)
    preds = preds / temperature

    preds = preds - np.max(preds)

    exp_preds = np.exp(preds)
    return exp_preds / np.sum(exp_preds)

## 7. Transformer Model

In [38]:
class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim):
        super().__init__()

        self.att = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim // num_heads,
            dropout=0.2
        )

        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dropout(0.2),
            layers.Dense(embed_dim),
        ])

        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)

        self.dropout = layers.Dropout(0.2)

    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]

        causal_mask = tf.linalg.band_part(
            tf.ones((seq_len, seq_len)), -1, 0
        )

        attn = self.att(inputs, inputs, attention_mask=causal_mask)
        attn = self.dropout(attn, training=training)

        x = self.norm1(inputs + attn)

        ffn = self.ffn(x)
        ffn = self.dropout(ffn, training=training)

        return self.norm2(x + ffn)

In [39]:
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()

        self.token_emb = layers.Embedding(vocab_size, embed_dim)
        self.pos_emb = layers.Embedding(maxlen, embed_dim)
        self.dropout = layers.Dropout(0.2)

    def call(self, x):
        seq_len = tf.shape(x)[1]

        positions = tf.range(seq_len)

        x = self.token_emb(x)
        pos = self.pos_emb(positions)

        return self.dropout(x + pos)

In [41]:
embed_dim = 256
num_heads = 8
ff_dim = 512

inputs = layers.Input(shape=(seq_length,))

x = TokenAndPositionEmbedding(seq_length, vocab_size, embed_dim)(inputs)

x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)  # 2 blockia

x = layers.GlobalAveragePooling1D()(x)

x = layers.Dense(256, activation="gelu")(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(vocab_size, activation="softmax")(x)

transformer_model = keras.Model(inputs, outputs)

transformer_model.compile(
    loss=keras.losses.SparseCategoricalCrossentropy(),
    optimizer=keras.optimizers.Adam(learning_rate=3e-4, clipnorm=1.0),
    metrics=["accuracy"]
)

## 8. Train

In [42]:
history_t = transformer_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=12,
    batch_size=64,
    callbacks=[callback]
)

Epoch 1/12
658/658 [==============================] - 15s 17ms/step - loss: 6.9343 - accuracy: 0.0676 - val_loss: 6.8583 - val_accuracy: 0.0595
Epoch 2/12
658/658 [==============================] - 7s 11ms/step - loss: 6.6432 - accuracy: 0.0690 - val_loss: 6.9018 - val_accuracy: 0.0595
Epoch 3/12
658/658 [==============================] - 7s 10ms/step - loss: 6.5935 - accuracy: 0.0698 - val_loss: 6.8653 - val_accuracy: 0.0595
Epoch 4/12
658/658 [==============================] - 7s 10ms/step - loss: 6.5075 - accuracy: 0.0704 - val_loss: 7.0321 - val_accuracy: 0.0595


## 9. Text Generation

In [43]:
def predict_next(text, model, temperature=1.0):
    seq = tokenizer.texts_to_sequences([text])[0]
    seq = seq[-seq_length:]

    if len(seq) < seq_length:
        seq = [0] * (seq_length - len(seq)) + seq

    preds = model.predict(np.array([seq]), verbose=0)[0]
    return sample_probs(preds, temperature)


def generate_text(seed_text, model, num_words=20, temperature=0.8):
    text = seed_text

    for _ in range(num_words):
        probs = predict_next(text, model, temperature)

        next_idx = np.random.choice(len(probs), p=probs)

        next_word = index_to_word.get(next_idx, "<UNK>")
        text += " " + next_word

    return text

In [44]:
print(generate_text("the meaning of life", transformer_model, 20))

the meaning of life and unk were record the the richest , of of . 's , the baptismal , the when unk of
